In [8]:
from evogym import sample_robot
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import copy
import matplotlib.pyplot as plt
from matplotlib import animation
import gymnasium as gym
import evogym.envs
from evogym import sample_robot
from evogym.utils import get_full_connectivity
from evogym import is_connected, has_actuator
from tqdm import tqdm
from datetime import datetime
import json
import os

# Base functions

In [9]:
class Network(nn.Module):
    def __init__(self, n_in, h_size, n_out):
        super().__init__()
        self.fc1 = nn.Linear(n_in, h_size)
        self.fc2 = nn.Linear(h_size, h_size)
        self.fc3 = nn.Linear(h_size, n_out)
 
        self.n_out = n_out

    def reset(self):
        pass
    
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)

        x = self.fc2(x)
        x = F.relu(x)

        x = self.fc3(x)
        return x

class Agent:
    def __init__(self, Net, config, genes = None):
        self.config = config
        self.Net = Net
        self.model = None
        self.fitness = None

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu")

        self.make_network()
        if genes is not None:
            self.genes = genes

    def __repr__(self):  # pragma: no cover
        return f"Agent {self.model} > fitness={self.fitness}"

    def __str__(self):  # pragma: no cover
        return self.__repr__()

    def make_network(self):
        n_in = self.config["n_in"]
        h_size = self.config["h_size"]
        n_out = self.config["n_out"]
        self.model = self.Net(n_in, h_size, n_out).to(self.device).double()
        return self

    @property
    def genes(self):
        if self.model is None:
            return None
        with torch.no_grad():
            params = self.model.parameters()
            vec = torch.nn.utils.parameters_to_vector(params)
        return vec.cpu().double().numpy()

    @genes.setter
    def genes(self, params):
        if self.model is None:
            self.make_network()
        assert len(params) == len(
            self.genes), "Genome size does not fit the network size"
        if np.isnan(params).any():
            raise
        a = torch.tensor(params, device=self.device)
        torch.nn.utils.vector_to_parameters(a, self.model.parameters())
        self.model = self.model.to(self.device).double()
        self.fitness = None
        return self

    def mutate_ga(self):
        genes = self.genes
        n = len(genes)
        f = np.random.choice([False, True], size=n, p=[1/n, 1-1/n])
        
        new_genes = np.empty(n)
        new_genes[f] = genes[f]
        noise = np.random.randn(n-sum(f))
        new_genes[~f] = noise
        return new_genes

    def act(self, obs):
        # continuous actions
        with torch.no_grad():
            x = torch.tensor(obs).double().unsqueeze(0).to(self.device)
            actions = self.model(x).cpu().detach().numpy()
        return actions


In [24]:
base = np.array([
    [3, 3, 3, 3, 3],
    [3, 3, 3, 0, 3],
    [3, 3, 0, 3, 3],
    [3, 3, 0, 3, 3],
    [3, 3, 0, 3, 3]
    ])

def make_env(env_name, robot=None, seed=None, **kwargs):
    if robot is None: 
        env = gym.make(env_name)
    else:
        connections = get_full_connectivity(robot)
        env = gym.make(env_name, body=robot)
    env.robot = robot
    if seed is not None:
        env.seed(seed)
        
    return env

def evaluate(agent, env, max_steps=500, render=False):
    obs, i = env.reset()
    agent.model.reset()
    reward = 0
    steps = 0
    done = False
    while not done and steps < max_steps:
        if render:
            env.render()
        action = agent.act(obs)
        obs, r, done, trunc, _ = env.step(action)
        reward += r
        steps += 1
    return reward

def get_cfg(env_name, robot=None):
    env = make_env(env_name, robot=base)
    cfg = {
        "n_in": env.observation_space.shape[0],
        "h_size": 32,
        "n_out": env.action_space.shape[0],
    }
    env.close()
    return cfg

def mp_eval(a, cfg):
    env = make_env(cfg["env_name"], robot=cfg["robot"])
    fit = evaluate(a, env, max_steps=cfg["max_steps"])
    env.close()
    return fit

In [25]:
def save_solution(a, cfg, base_dir="results"):
    save_cfg = {}
    for i in ["env_name", "robot", "n_in", "h_size", "n_out"]:
        assert i in cfg, f"{i} not in config"
        save_cfg[i] = cfg[i]
        
    save_cfg["robot"] = cfg["robot"].tolist()
    save_cfg["genes"] = a.genes.tolist()
    save_cfg["fitness"] = float(a.fitness)

    date_heure = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    env_name = cfg["env_name"]
    target_dir = os.path.join(base_dir, f"{env_name}_{date_heure}")
    os.makedirs(target_dir, exist_ok=True) 
    fitness_str = f"{save_cfg['fitness']:.2f}" 
    filename = f"{fitness_str}.json"
    full_path = os.path.join(target_dir, filename)
    
    with open(full_path, "w") as f:
        json.dump(save_cfg, f, indent=4) 
    return save_cfg

def load_solution(name):
    with open(name, "r") as f:
        cfg = json.load(f)
    cfg["robot"] = np.array(cfg["robot"])
    cfg["genes"] = np.array(cfg["genes"])
    a = Agent(Network, cfg, genes=cfg["genes"])
    a.fitness = cfg["fitness"]
    return a

# Evolution

In [26]:
walker = np.array([
    [3, 3, 3, 3, 3],
    [3, 3, 3, 0, 3],
    [3, 3, 0, 3, 3],
    [3, 3, 0, 3, 3],
    [3, 3, 0, 3, 3]
    ])

env_name = 'Walker-v0'
robot = walker

cfg = get_cfg(env_name, robot)
a = Agent(Network, cfg)

### Evolution Strategy

In [27]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import Parallel, delayed

# ---------------------------------------------------------
# 1. Fonction Worker (Évaluation d'un individu)
# ---------------------------------------------------------
def evaluate_worker(genes, cfg):
    """
    Cette fonction est exécutée en parallèle par Joblib.
    Chaque processus crée son propre environnement pour éviter les conflits.
    """
    # Création de l'environnement local
    env = make_env(cfg["env_name"], robot=cfg["robot"])
    
    # Instanciation de l'agent avec les gènes proposés
    ind = Agent(Network, cfg, genes=genes)
    
    # Évaluation de l'agent
    fitness = evaluate(ind, env, max_steps=cfg["max_steps"])
    
    env.close()
    return fitness


# ---------------------------------------------------------
# 2. Algorithme Principal (Stratégie d'Évolution)
# ---------------------------------------------------------
def ES(config):
    # Récupération et fusion des configurations
    cfg = get_cfg(config["env_name"], robot=config["robot"])
    cfg = {**config, **cfg}
    
    mu = cfg["mu"]
    lam = cfg["lambda"] # 'lambda' est un mot-clé réservé, on utilise 'lam'
    
    # Initialisation des poids pour la mise à jour
    w = np.array([np.log(mu + 0.5) - np.log(i) for i in range(1, mu + 1)])
    w /= np.sum(w)
    
    # Création d'un environnement temporaire juste pour initialiser l'élite
    tmp_env = make_env(cfg["env_name"], robot=cfg["robot"])
    elite = Agent(Network, cfg)
    elite.fitness = -np.inf
    theta = elite.genes
    d = len(theta)
    tmp_env.close()

    fits = []
    total_evals = []

    bar = tqdm(range(cfg["generations"]))
    for gen in bar:
        
        # ---------------------------------------------------
        # ÉTAPE 1 : Vectorisation de la génération des gènes
        # ---------------------------------------------------
        # On génère une matrice de bruit de taille (lambda, d)
        noise = np.random.randn(lam, d)
        # On applique le bruit à theta pour créer toute la population d'un coup
        population_genes = theta + noise * cfg["sigma"]


        # ---------------------------------------------------
        # ÉTAPE 2 : Évaluation en parallèle avec Joblib
        # ---------------------------------------------------
        # n_jobs=-1 indique à Joblib d'utiliser TOUS les cœurs de ton CPU.
        # delayed() encapsule la fonction pour qu'elle soit exécutée plus tard par les workers.
        pop_fitness = Parallel(n_jobs=-1)(
            delayed(evaluate_worker)(genes, cfg) for genes in population_genes
        )
        
        
        # ---------------------------------------------------
        # ÉTAPE 3 : Sélection et mise à jour
        # ---------------------------------------------------
        # Tri des indices du meilleur au pire fitness
        inv_fitnesses = [-f for f in pop_fitness]
        idx = np.argsort(inv_fitnesses)
        
        # On récupère les gènes des 'mu' meilleurs individus
        top_genes = population_genes[idx[:mu]]
        
        # Calcul de la direction (step) et mise à jour de theta
        step = np.zeros(d)
        for i in range(mu):
            step += w[i] * (top_genes[i] - theta)
            
        theta = theta + step * cfg["lr"]

        # Mise à jour de l'agent élite global
        if pop_fitness[idx[0]] > elite.fitness:
            # On utilise .copy() pour éviter de lier les références mémoire
            elite.genes = population_genes[idx[0]].copy()
            elite.fitness = pop_fitness[idx[0]]

        # Suivi des statistiques
        fits.append(elite.fitness)
        total_evals.append(lam * (gen + 1))

        bar.set_description(f"Best: {elite.fitness:.2f}")
        
    # Affichage de la courbe d'apprentissage
    plt.plot(total_evals, fits)
    plt.xlabel("Evaluations")
    plt.ylabel("Fitness")
    plt.title("Progression de l'algorithme ES")
    plt.grid(True)
    plt.show()
    
    return elite

In [28]:
config = {
    "env_name": "Walker-v0",
    "robot": walker,
    "generations": 500, 
    "lambda": 20,  # Population size
    "mu": 5, # Parents pop size
    "sigma": 0.1, # mutation std
    "lr": 1, # Learning rate
    "max_steps": 500, 
}

a = ES(config)
a.fitness

  0%|          | 0/500 [00:01<?, ?it/s]


NameNotFound: Environment `Walker` doesn't exist. Did you mean: `Walker2d`?

In [ ]:
cfg = {**config, **cfg}
save_solution(a, cfg)